In [1]:
from Ingestion.pdf_to_markdown import PDF2Markdown

In [3]:
pdf = "data/raw_pdf/2025_Amazon.pdf"

In [4]:
from pathlib import Path

In [5]:
pdf_path = Path(pdf)

In [6]:
pdf_path

WindowsPath('data/raw_pdf/2025_Amazon.pdf')

In [15]:
pdf_path.stem

'2025_Amazon'

In [ ]:
input_dir = "data/raw_pdf"

In [ ]:
Path(input_dir).glob("*.pdf")

In [ ]:

    prompt = build_extraction_prompt(
        company=company,
        year=year,
        context=context
    )

    structured_llm = groq_llm.with_structured_output(FinancialSchema)
    metrics = structured_llm.invoke(prompt)

    return metrics.model_dump()

In [ ]:
from fastapi import APIRouter, HTTPException
from pydantic import BaseModel
from llm.LLM import groq_llm
from vector_store.retriever import get_retriever
import asyncio

router = APIRouter()

llm = groq_llm

class ChatSchema(BaseModel):
    company: str
    year: str
    inquiry: str

def chat(request: ChatSchema, retriever = get_retriever):
    try:
        search_query = f"{request.company} {request.year} {request.inquiry}"
        print(f"[1]")
        retriever = get_retriever()
        docs = retriever.invoke(search_query)
        print(f"[2]")
        context = "\n\n".join(doc.page_content for doc in docs[:2])
        print(f"[3]")

        prompt = f"""
        Use the following context from corporate reports to answer the user's question.
        If the context does not contain relevant information,
        politely indicate that you did not have enough data.

        Company: {request.company}
        Year: {request.year}

        Context:
        {context}

        User Question: {request.inquiry}

        Answer:
        """
        print(f"[4]")

        results = llm.invoke(prompt)
        print(f"[5]")
        output = results.content
        return {"answer": output}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [38]:
chatt = ChatSchema(company="Apple", year="2025", inquiry="total revenue of the apply!")

In [39]:
result = chat(request= chatt)

[1]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6042.74it/s]


[2]
[3]
[4]
[5]


In [40]:
print(result)

{'answer': '**Apple’s total revenue (net sales)**  \n\n- **Fiscal year\u202f2025:**\u202f**$416,161\u202fmillion** (≈\u202f$416.2\u202fbillion)  \n- **Fiscal year\u202f2024 (for reference):**\u202f$391,035\u202fmillion (≈\u202f$391.0\u202fbillion)\n\nThese figures are taken from Apple’s 2025 Form\u202f10‑K, Note\u202f2 – Revenue, which reports “Total net sales” for each year.'}
